# NB03 - Data Maps: Training Dynamics
## Background
Dataset cartography (Swayamdipta et al., 2020) characterizes each training example by how a model behaves on it across training epochs — not just the final prediction but the trajectory. Three regions emerge: easy-to-learn (consistently predicted correctly), hard-to-learn (rarely correct), and ambiguous (variable correctness). These regions have distinct data quality profiles: easy samples may be redundant, hard samples may be mislabeled or genuinely difficult, and ambiguous samples are the most informative for training.

Swayamdipta et al. showed that training on ambiguous samples alone often outperforms training on the full dataset, and that hard samples have higher overlap with mislabeled examples.

## What You'll Learn
- Tracking model confidence on each sample across training epochs
- Confidence mean and variability as 2D cartography axes
- Identifying easy, ambiguous, and hard sample regions
- Correlating training dynamics with ground truth label noise

## Why This Matters
Data maps turn training dynamics into a debugging tool. High variability + low confidence strongly predicts mislabeled examples. Training only on ambiguous samples can reduce dataset size by 30-50% while maintaining performance.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from typing import List, Dict, Tuple

# ── Minimal logistic regression for tracking dynamics ─────────────────────
class LogisticRegression:
    def __init__(self, n_features: int, n_classes: int, lr: float = 0.1):
        self.W = np.random.randn(n_features, n_classes) * 0.1
        self.b = np.zeros(n_classes)
        self.lr = lr

    def forward(self, X: np.ndarray) -> np.ndarray:
        logits = X @ self.W + self.b
        logits -= logits.max(axis=1, keepdims=True)
        exp_l = np.exp(logits)
        return exp_l / exp_l.sum(axis=1, keepdims=True)

    def backward(self, X: np.ndarray, y: np.ndarray) -> float:
        probs = self.forward(X)
        n = len(y)
        loss = -np.mean(np.log(probs[np.arange(n), y] + 1e-10))
        dlogits = probs
        dlogits[np.arange(n), y] -= 1
        dlogits /= n
        self.W -= self.lr * X.T @ dlogits
        self.b -= self.lr * dlogits.sum(axis=0)
        return loss

# ── Generate dataset with known noisy labels ──────────────────────────────
np.random.seed(42)
n_samples, n_classes, n_features = 600, 4, 8
n_per_class = n_samples // n_classes

class_means = np.random.randn(n_classes, n_features) * 2
X_all, y_true = [], []
for c in range(n_classes):
    feats = class_means[c] + np.random.randn(n_per_class, n_features)
    X_all.append(feats); y_true.extend([c] * n_per_class)
X_all, y_true = np.vstack(X_all), np.array(y_true)

# Label noise: 15% mislabeled
y_noisy = y_true.copy()
noisy_idx = np.random.choice(n_samples, int(n_samples * 0.15), replace=False)
y_noisy[noisy_idx] = (y_true[noisy_idx] + 1) % n_classes

print(f'Dataset: {n_samples} samples, {n_features} features, {n_classes} classes')
print(f'Mislabeled: {len(noisy_idx)} samples ({100*len(noisy_idx)/n_samples:.0f}%)')

In [ ]:
# ── Train and record per-sample confidence over epochs ────────────────────
np.random.seed(0)
model = LogisticRegression(n_features, n_classes, lr=0.05)
n_epochs = 30

# Track confidence on noisy labels per sample per epoch
confidence_history = np.zeros((n_epochs, n_samples))

batch_size = 64
for epoch in range(n_epochs):
    # Shuffle and train
    perm = np.random.permutation(n_samples)
    for i in range(0, n_samples, batch_size):
        idx = perm[i:i+batch_size]
        model.backward(X_all[idx], y_noisy[idx])
    # Record confidence on each sample for its noisy label
    all_probs = model.forward(X_all)
    for s in range(n_samples):
        confidence_history[epoch, s] = all_probs[s, y_noisy[s]]

# Cartography metrics per sample
conf_mean = confidence_history.mean(axis=0)
conf_std  = confidence_history.std(axis=0)

# Categorize
easy_mask      = (conf_mean > 0.7) & (conf_std < 0.15)
hard_mask      = (conf_mean < 0.3)
ambiguous_mask = (conf_std > 0.15) & ~hard_mask

print('Dataset map regions:')
print(f'  Easy-to-learn:   {easy_mask.sum()} samples')
print(f'  Ambiguous:       {ambiguous_mask.sum()} samples')
print(f'  Hard-to-learn:   {hard_mask.sum()} samples')

# What fraction of hard samples are mislabeled?
noisy_set = set(noisy_idx)
hard_indices = np.where(hard_mask)[0]
hard_noisy_pct = len([i for i in hard_indices if i in noisy_set]) / max(len(hard_indices), 1)
print(f'  Mislabeled in hard region: {100*hard_noisy_pct:.1f}%')
easy_indices = np.where(easy_mask)[0]
easy_noisy_pct = len([i for i in easy_indices if i in noisy_set]) / max(len(easy_indices), 1)
print(f'  Mislabeled in easy region: {100*easy_noisy_pct:.1f}%')

In [ ]:
# ── Dataset map visualization ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Main dataset map
is_noisy = np.isin(np.arange(n_samples), noisy_idx)
colors = np.where(is_noisy, 'red', 'blue')

axes[0].scatter(conf_mean[~is_noisy], conf_std[~is_noisy],
                c='blue', alpha=0.4, s=15, label='Clean label')
axes[0].scatter(conf_mean[is_noisy], conf_std[is_noisy],
                c='red', alpha=0.6, s=25, marker='x', label='Noisy label')

# Region annotations
axes[0].axvline(0.7, color='gray', linestyle=':', alpha=0.5)
axes[0].axvline(0.3, color='gray', linestyle=':', alpha=0.5)
axes[0].axhline(0.15, color='gray', linestyle=':', alpha=0.5)
axes[0].text(0.85, 0.05, 'Easy', fontsize=12, color='green', transform=axes[0].transAxes)
axes[0].text(0.05, 0.85, 'Hard', fontsize=12, color='red', transform=axes[0].transAxes)
axes[0].text(0.35, 0.85, 'Ambiguous', fontsize=12, color='orange', transform=axes[0].transAxes)
axes[0].set_title('Dataset Map (Swayamdipta et al., 2020)')
axes[0].set_xlabel('Mean confidence across epochs')
axes[0].set_ylabel('Confidence variability (std)')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# Confidence trajectories for example samples
easy_ex   = np.where(easy_mask)[0][:3]
hard_ex   = np.where(hard_mask)[0][:3]
ambig_ex  = np.where(ambiguous_mask)[0][:3]

epochs = range(n_epochs)
for i, idx in enumerate(easy_ex):
    axes[1].plot(epochs, confidence_history[:, idx], 'g-', alpha=0.5,
                 label='Easy' if i==0 else '')
for i, idx in enumerate(hard_ex):
    axes[1].plot(epochs, confidence_history[:, idx], 'r-', alpha=0.5,
                 label='Hard' if i==0 else '')
for i, idx in enumerate(ambig_ex):
    axes[1].plot(epochs, confidence_history[:, idx], 'orange', alpha=0.5,
                 label='Ambiguous' if i==0 else '')

axes[1].set_title('Confidence Trajectories per Region')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('P(given label)')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{BASE}/s29_03_data_maps.png', dpi=100, bbox_inches='tight')
plt.show()